# Data Engineering et Machine Learning avec Snowflake

Ce notebook presente un pipeline complet dans Snowflake pour l'ingestion, la preparation des donnees, l'entrainement de modeles, l'inference et l'exposition du modele via Streamlit.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn
import xgboost

# Recuperer la session active Snowflake
session = get_active_session()

package_rows = [
    ("snowpark", "OK"),
    ("pandas", pd.__version__),
    ("numpy", np.__version__),
    ("scikit-learn", sklearn.__version__),
    ("matplotlib", matplotlib.__version__),
    ("seaborn", seaborn.__version__),
    ("xgboost", xgboost.__version__),
]

print("=" * 58)
print("Initialisation de l'environnement Snowflake")
print("=" * 58)
for package_name, package_version in package_rows:
    print(f"{package_name:<14} -> {package_version}")

print("")
print(f"Base active   : {session.get_current_database()}")
print(f"Schema actif  : {session.get_current_schema()}")
print(f"Warehouse     : {session.get_current_warehouse()}")
print("Environnement valide, vous pouvez lancer le pipeline.")


## 1. Ingestion des donnees

Dans cette etape, nous creons le format de fichier, le stage et la table brute pour charger le dataset JSON depuis S3.

In [ ]:
sql_statements = [
    """
    CREATE OR REPLACE FILE FORMAT HOUSE_JSON_FORMAT
      TYPE = 'JSON'
      STRIP_OUTER_ARRAY = TRUE
    """,
    """
    CREATE OR REPLACE STAGE HOUSE_STAGE
      URL = 's3://logbrain-datalake/datasets/house_price/'
      FILE_FORMAT = HOUSE_JSON_FORMAT
    """,
    """
    CREATE OR REPLACE TABLE HOUSE_PRICES_RAW (
      raw_data VARIANT
    )
    """,
    """
    COPY INTO HOUSE_PRICES_RAW
      FROM @HOUSE_STAGE/Housing_Price_Data.json
      FILE_FORMAT = HOUSE_JSON_FORMAT
    """
]

for statement in sql_statements:
    session.sql(statement).collect()

print("Ingestion terminee dans HOUSE_PRICES_RAW.")
print("Apercu du contenu brut :")
session.sql("SELECT raw_data FROM HOUSE_PRICES_RAW LIMIT 2").show()

print("Controle volumetrique apres ingestion :")
session.sql("""
SELECT
  COUNT(*) AS nb_lignes_chargees,
  COUNT(raw_data) AS nb_variants_non_null
FROM HOUSE_PRICES_RAW
""").show()


## 2. Creation de la table structuree

Les donnees brutes stockees en `VARIANT` sont converties en une table relationnelle typee, plus simple a explorer et a exploiter pour le machine learning.

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE HOUSE_PRICES AS
SELECT
  raw_data:price::NUMBER             AS price,
  raw_data:area::NUMBER              AS area,
  raw_data:bedrooms::NUMBER          AS bedrooms,
  raw_data:bathrooms::NUMBER         AS bathrooms,
  raw_data:stories::NUMBER           AS stories,
  raw_data:mainroad::VARCHAR         AS mainroad,
  raw_data:guestroom::VARCHAR        AS guestroom,
  raw_data:basement::VARCHAR         AS basement,
  raw_data:hotwaterheating::VARCHAR  AS hotwaterheating,
  raw_data:airconditioning::VARCHAR  AS airconditioning,
  raw_data:parking::NUMBER           AS parking,
  raw_data:prefarea::VARCHAR         AS prefarea,
  raw_data:furnishingstatus::VARCHAR AS furnishingstatus
FROM HOUSE_PRICES_RAW
""").collect()

print("Table HOUSE_PRICES creee et typee.")
print("Extrait de la table structuree :")
session.sql("SELECT * FROM HOUSE_PRICES LIMIT 4").show()

print("Controle rapide des informations cles :")
session.sql("""
SELECT
  COUNT(*) AS nb_lignes,
  MIN(price) AS prix_min,
  MAX(price) AS prix_max,
  AVG(area) AS surface_moyenne
FROM HOUSE_PRICES
""").show()


## 3. Exploration initiale

Nous chargeons la table Snowflake dans un DataFrame Snowpark afin d'observer rapidement la volumetrie et les colonnes disponibles.

In [ ]:
# Charger la table dans un DataFrame Snowpark
df = session.table("HOUSE_PRICES")

row_count = df.count()
column_count = len(df.columns)
numeric_candidates = ["PRICE", "AREA", "BEDROOMS", "BATHROOMS", "STORIES", "PARKING"]
categorical_candidates = [col_name for col_name in df.columns if col_name not in numeric_candidates]

print(f"Volume du dataset : {row_count} lignes")
print(f"Nombre de colonnes : {column_count}")
print(f"Variables numeriques attendues : {numeric_candidates}")
print(f"Variables plutot categorielles : {categorical_candidates}")
print(f"Ordre complet des colonnes : {df.columns}")


## 4. Apercu des donnees

Cette section permet de visualiser les premieres lignes et les statistiques descriptives du dataset.

In [ ]:
print("=== Echantillon du dataset ===")
df.limit(5).show()

print("\n=== Resume statistique Snowpark ===")
df.describe().show()

print("\n=== Vue ciblee sur quelques colonnes metier ===")
session.sql("""
SELECT price, area, bedrooms, bathrooms, furnishingstatus
FROM HOUSE_PRICES
LIMIT 5
""").show()


## 5. Controle des valeurs manquantes

Avant toute preparation ou modelisation, nous verifions la presence eventuelle de valeurs nulles dans chaque colonne.

In [ ]:
from snowflake.snowpark.functions import col, count, when

# Calculer les valeurs nulles par colonne
null_counts = df.select([
    count(when(col(column_name).isNull(), column_name)).alias(column_name)
    for column_name in df.columns
])

print("=== Controle des valeurs nulles ===")
null_counts.show()
print("Lecture attendue : une ligne avec le nombre de nulls pour chaque colonne.")


## 6. Distribution de la variable cible

Nous etudions la distribution du prix des maisons avec un histogramme et un boxplot.

In [ ]:
import matplotlib.pyplot as plt

# Conversion vers Pandas pour la visualisation
df_pd = df.to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(15, 5), facecolor="#f7f4ee")

axes[0].hist(df_pd["PRICE"], bins=26, color="#3d5a80", edgecolor="white", alpha=0.95)
axes[0].set_title("Repartition du prix des maisons")
axes[0].set_xlabel("Prix")
axes[0].set_ylabel("Nombre d'observations")
axes[0].grid(alpha=0.18)

axes[1].boxplot(
    df_pd["PRICE"],
    patch_artist=True,
    boxprops=dict(facecolor="#ee6c4d", color="#293241"),
    medianprops=dict(color="white", linewidth=2),
    whiskerprops=dict(color="#293241"),
    capprops=dict(color="#293241")
)
axes[1].set_title("Dispersion et valeurs extremes")
axes[1].set_ylabel("Prix")

plt.tight_layout()
plt.show()


## 7. Correlations entre variables numeriques

Une matrice de correlation permet d'identifier les relations lineaires entre les variables quantitatives.

In [ ]:
import seaborn as sns

num_cols = ["PRICE", "AREA", "BEDROOMS", "BATHROOMS", "STORIES", "PARKING"]
corr_matrix = df_pd[num_cols].corr()

plt.figure(figsize=(9, 6), facecolor="#f7f4ee")
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="crest",
    linewidths=0.6,
    square=True
)
plt.title("Matrice des correlations numeriques")
plt.tight_layout()
plt.show()


## 8. Analyse des variables categorielles

Nous observons la distribution des variables qualitatives du dataset.

In [ ]:
cat_cols = [
    "MAINROAD", "GUESTROOM", "BASEMENT",
    "HOTWATERHEATING", "AIRCONDITIONING",
    "PREFAREA", "FURNISHINGSTATUS"
]

color_cycle = ["#3d5a80", "#98c1d9", "#e0fbfc", "#ee6c4d", "#bc6c25", "#6c757d", "#84a98c"]
fig, axes = plt.subplots(2, 4, figsize=(18, 8), facecolor="#f7f4ee")
axes = axes.flatten()

for i, col_name in enumerate(cat_cols):
    counts = df_pd[col_name].value_counts().sort_values(ascending=False)
    axes[i].bar(counts.index.astype(str), counts.values, color=color_cycle[i], edgecolor="white")
    axes[i].set_title(f"Distribution de {col_name.lower()}")
    axes[i].set_ylabel("Effectif")
    axes[i].tick_params(axis="x", rotation=18)

axes[-1].set_visible(False)

plt.suptitle("Vue d'ensemble des variables categorielles", fontsize=14)
plt.tight_layout()
plt.show()


## 9. Encodage des variables

Les colonnes categorielles utiles au modele sont converties en valeurs numeriques.

In [ ]:
import pandas as pd

df_pd = df.to_pandas()

binary_cols = ["MAINROAD", "GUESTROOM", "BASEMENT",
               "HOTWATERHEATING", "AIRCONDITIONING", "PREFAREA"]

for feature_name in binary_cols:
    df_pd[feature_name] = df_pd[feature_name].map({"yes": 1, "no": 0})

df_pd["FURNISHINGSTATUS"] = df_pd["FURNISHINGSTATUS"].map({
    "furnished": 2,
    "semi-furnished": 1,
    "unfurnished": 0
})

print("Encodage termine.")
print("Apercu des donnees encodees :")
print(df_pd.head())
print(f"\nTypes des colonnes apres transformation :\n{df_pd.dtypes}")


## 10. Separation des variables explicatives et de la cible

Le dataset est separe entre les features `X` et la variable cible `y`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Definir la cible
y = df_pd['PRICE']

# Conserver toutes les autres colonnes comme variables explicatives
X = df_pd.drop(columns=['PRICE'])

print(f"Matrice X : {X.shape}")
print(f"Vecteur cible y : {y.shape}")
print(f"\nVariables retenues : {list(X.columns)}")


## 11. Preparation train / test

Nous construisons les jeux d'entrainement et de test, puis nous appliquons une normalisation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train, 20% test
    random_state=42
)

# Standardisation des variables
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Dimensions de X_train : {X_train_scaled.shape}")
print(f"Dimensions de X_test  : {X_test_scaled.shape}")
print("\nRepartition du jeu de donnees :")
print(f"   Entrainement : {len(X_train)} lignes ({len(X_train)/len(X)*100:.0f}%)")
print(f"   Test         : {len(X_test)} lignes ({len(X_test)/len(X)*100:.0f}%)")


## 12. Entrainement des modeles

Trois modeles de regression sont entraines pour comparer leurs performances sur ce cas d'usage.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Definir les modeles a comparer
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42)
}

# Entrainer les modeles un par un
trained_models = {}
for model_name, estimator in models.items():
    estimator.fit(X_train_scaled, y_train)
    trained_models[model_name] = estimator
    print(f"Apprentissage termine pour : {model_name}")

print("\nLes trois modeles ont ete entraines.")


## 13. Evaluation des performances

Les modeles sont compares a l'aide de metriques adaptees a la regression : MAE, RMSE et R2.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results = []

for model_name, estimator in trained_models.items():
    predictions = estimator.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Modele": model_name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2": round(r2, 4)
    })

    print(f"Resultats pour {model_name}")
    print(f"   - MAE  : {mae:,.0f}")
    print(f"   - RMSE : {rmse:,.0f}")
    print(f"   - R2   : {r2:.4f}\n")

# Construire le tableau recapitulatif
results_df = pd.DataFrame(results)
print("=== Tableau de comparaison des modeles ===")
print(results_df.to_string(index=False))


## 14. Visualisation comparative

Les metriques sont representees graphiquement pour faciliter l'interpretation des resultats.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor="#f7f4ee")
metrics = ["MAE", "RMSE", "R2"]
colors = ["#3d5a80", "#ee6c4d", "#84a98c"]

for i, metric in enumerate(metrics):
    axes[i].bar(results_df["Modele"], results_df[metric],
                color=colors[i], edgecolor="white")
    axes[i].set_title(f"Comparaison par {metric}")
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis="x", rotation=15)
    axes[i].grid(axis="y", alpha=0.15)

plt.suptitle("Lecture visuelle des metriques de performance", fontsize=14)
plt.tight_layout()
plt.show()


## 15. Optimisation par recherche d'hyperparametres

Un `GridSearchCV` est lance pour ameliorer les performances du modele XGBoost.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Parametres reduits pour rester leger en memoire
param_grid = {
    'xgb__n_estimators': [100, 200],
    'xgb__max_depth': [3, 5],
    'xgb__learning_rate': [0.1, 0.2],
}

pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(random_state=42))
])

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=1,
    verbose=1
)

print("Lancement du GridSearchCV...")
grid_search.fit(X_train, y_train)

print(f"\nParametres optimaux : {grid_search.best_params_}")
print(f"Meilleur score R2 en validation croisee : {grid_search.best_score_:.4f}")


## 16. Evaluation du meilleur modele

Nous analysons les performances du modele optimise et nous les comparons au modele de base.

In [ ]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

mae_best = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best = r2_score(y_test, y_pred_best)

print("=== Evaluation du pipeline XGBoost optimise ===")
print(f"   MAE  : {mae_best:,.0f}")
print(f"   RMSE : {rmse_best:,.0f}")
print(f"   R2   : {r2_best:.4f}")

# Comparaison avec le modele XGBoost initial
xgb_base = trained_models['XGBoost']
y_pred_base = xgb_base.predict(X_test_scaled)
r2_base = r2_score(y_test, y_pred_base)

print("\nComparaison avant / apres optimisation :")
print(f"   XGBoost initial   -> R2 : {r2_base:.4f}")
print(f"   XGBoost optimise  -> R2 : {r2_best:.4f}")
improvement = (r2_best - r2_base) * 100
print(f"   Ecart observe     -> {improvement:+.2f}%")


## 17. Comparaison predictions / realite

Un nuage de points permet de visualiser la proximite entre les valeurs reelles et les valeurs predites.

In [ ]:
plt.figure(figsize=(8, 6), facecolor="#f7f4ee")
plt.scatter(y_test, y_pred_best, alpha=0.55, color="#3d5a80", edgecolors="white", s=46)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         linestyle="--", color="#ee6c4d", linewidth=2, label="Alignement parfait")
plt.xlabel("Prix reel")
plt.ylabel("Prix predit")
plt.title("Comparaison entre les valeurs reelles et predites")
plt.legend()
plt.tight_layout()
plt.show()


## 18. Enregistrement dans le Model Registry

Le meilleur modele est enregistre dans le Snowflake Model Registry avec ses metriques principales.

In [ ]:
from snowflake.ml.registry import Registry

# Creer le registry
registry = Registry(session=session)

model_name = "house_price_xgboost"
version_name = "v1"

# Enregistrer le meilleur modele (pipeline complet: scaler + XGBoost)
try:
    model_ref = registry.log_model(
        model=best_model,
        model_name=model_name,
        version_name=version_name,
        comment="Pipeline XGBoost optimise avec GridSearch",
        metrics={
            "r2": round(r2_best, 4),
            "mae": round(mae_best, 2),
            "rmse": round(rmse_best, 2)
        },
        sample_input_data=X_test.head(5)
    )
    print("Modele enregistre dans le Snowflake Model Registry.")
    print(f"   Nom du modele : {model_name}")
    print(f"   Version       : {version_name}")
    print(f"   R2            : {r2_best:.4f}")
except ValueError as err:
    if "already existed" in str(err):
        model_ref = registry.get_model(model_name).version(version_name)
        print("La version v1 existe deja dans le Model Registry.")
        print("La version existante est reutilisee pour la suite du notebook.")
        print(f"   Nom du modele : {model_name}")
        print(f"   Version       : {version_name}")
    else:
        raise


## 19. Verification du registry

Nous controlons que le modele enregistre est bien visible dans le registry.

In [ ]:
# Lister les modeles disponibles dans le registry
models_list = registry.show_models()
print("=== Contenu du Model Registry ===")
print(models_list)


## 20. Rechargement du modele

Le modele est recharge depuis le registry afin de preparer une etape d'inference.

In [ ]:
from snowflake.ml.registry import Registry
import pandas as pd

# Recharger le registry depuis la session
registry = Registry(session=session)

# Recuperer la version cible du modele
model_ref = registry.get_model("house_price_xgboost").version("v1")

print("Modele recupere depuis le registry.")


## 21. Inference sur de nouvelles donnees

Nous generons des predictions sur quelques maisons fictives pour valider le pipeline d'inference.

In [ ]:
# Construire un petit jeu d'exemples pour la prediction
new_houses = pd.DataFrame({
    'AREA': [200, 150, 300],
    'BEDROOMS': [3, 2, 4],
    'BATHROOMS': [2, 1, 3],
    'STORIES': [2, 1, 3],
    'MAINROAD': [1, 1, 0],
    'GUESTROOM': [0, 0, 1],
    'BASEMENT': [1, 0, 1],
    'HOTWATERHEATING': [0, 0, 1],
    'AIRCONDITIONING': [1, 0, 1],
    'PARKING': [2, 1, 3],
    'PREFAREA': [1, 0, 1],
    'FURNISHINGSTATUS': [2, 0, 1]
}).astype('int16')

# Le modele embarque deja les transformations necessaires
predictions = model_ref.run(new_houses, function_name="predict")

print("=== Resultats d'inference ===")
for idx, pred in enumerate(predictions['output_feature_0'], start=1):
    print(f"   Maison {idx} : {pred:,.0f} ")


## 22. Sauvegarde des predictions

Les resultats d'inference sont ecrits dans une table Snowflake dediee.

In [ ]:
# Ajouter les predictions au DataFrame source
new_houses['PREDICTED_PRICE'] = predictions['output_feature_0'].values

# Ecrire le resultat dans une table Snowflake
session.write_pandas(
    new_houses,
    table_name="HOUSE_PREDICTIONS",
    auto_create_table=True,
    overwrite=True
)

print("Les predictions ont ete sauvegardees dans HOUSE_PREDICTIONS.")

# Controler le contenu ecrit
session.table("HOUSE_PREDICTIONS").show()


## 23. Application Streamlit

Une interface Streamlit permet a un utilisateur metier de saisir des caracteristiques de bien immobilier et d'obtenir une estimation du prix.

In [ ]:
import streamlit as st
import pandas as pd
from streamlit.errors import StreamlitSetPageConfigMustBeFirstCommandError
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

# Dans Snowflake, la configuration de page peut deja etre definie par le runtime. Cette gestion reste donc sure et non bloquante.
try:
    st.set_page_config(
        page_title="House Price Studio",
        page_icon="home",
        layout="wide",
        initial_sidebar_state="expanded",
    )
except StreamlitSetPageConfigMustBeFirstCommandError:
    pass

st.markdown(
    """
    <style>
    :root {
      --bg: #f6f1ea;
      --card: rgba(255, 252, 247, 0.90);
      --ink: #243447;
      --muted: #6c757d;
      --line: #e4d8c8;
      --navy: #243447;
      --sand: #d9b382;
      --terracotta: #c97b63;
      --sage: #7b9e87;
    }

    html, body, [class*="css"] {
      font-family: "Georgia", "Segoe UI", sans-serif;
      color: var(--ink);
    }

    section.main > div {
      background:
        radial-gradient(circle at top left, rgba(217, 179, 130, 0.20), transparent 30%),
        radial-gradient(circle at top right, rgba(201, 123, 99, 0.16), transparent 24%),
        linear-gradient(180deg, #fcfaf6 0%, #f3ece4 100%);
    }

    div.block-container {
      max-width: 1280px;
      padding-top: 1.15rem;
      padding-bottom: 2rem;
    }

    div[data-testid="stSidebar"] {
      background: linear-gradient(180deg, #243447 0%, #35506b 100%);
    }

    div[data-testid="stSidebar"] * {
      color: #f8f5ef !important;
    }

    .hero {
      background: linear-gradient(135deg, #243447 0%, #35506b 54%, #c97b63 100%);
      color: white;
      border-radius: 24px;
      padding: 1.4rem 1.5rem;
      box-shadow: 0 16px 30px rgba(36, 52, 71, 0.18);
      margin-bottom: 1rem;
    }

    .panel {
      background: var(--card);
      border: 1px solid var(--line);
      border-radius: 20px;
      padding: 1rem 1.05rem;
      box-shadow: 0 12px 26px rgba(36, 52, 71, 0.07);
      margin-bottom: 0.85rem;
      backdrop-filter: blur(6px);
    }

    .panel, .panel-title {
      color: var(--ink);
    }

    .panel-title {
      font-weight: 700;
      font-size: 1.04rem;
      margin-bottom: 0.35rem;
    }

    .hint {
      color: var(--muted);
      font-size: 0.92rem;
    }

    .badge {
      display: inline-block;
      padding: 0.28rem 0.66rem;
      border-radius: 999px;
      background: #fff7ee;
      border: 1px solid #ead8bf;
      color: #7a4b2f;
      font-size: 0.82rem;
      margin: 0.1rem 0.25rem 0.1rem 0;
    }

    .result-box {
      background: linear-gradient(145deg, #fffdfa 0%, #f7ede2 100%);
      border: 1px solid #e8dccb;
      border-left: 10px solid var(--sand);
      border-radius: 20px;
      padding: 1.05rem;
      box-shadow: 0 14px 26px rgba(201, 123, 99, 0.10);
    }

    .result-box,
    .result-box h3,
    .result-box p,
    .result-box strong {
      color: var(--ink) !important;
    }

    div[data-testid="stButton"] > button {
      border-radius: 14px;
      border: none;
      padding: 0.72rem 0.95rem;
      font-weight: 700;
      background: linear-gradient(120deg, #c97b63 0%, #d9b382 100%);
      color: white;
      box-shadow: 0 10px 18px rgba(201, 123, 99, 0.22);
    }
    </style>
    """,
    unsafe_allow_html=True,
)

try:
    session = get_active_session()
except Exception:
    st.error("Session Snowflake introuvable. Executez cette application dans Snowflake Streamlit.")
    st.stop()

st.sidebar.markdown("## Informations de scoring")
st.sidebar.markdown("- Modele : house_price_xgboost")
st.sidebar.markdown("- Version : v1")
st.sidebar.markdown("- Sortie : estimation du prix")
st.sidebar.markdown("---")
st.sidebar.markdown("### Encodage utilise")
st.sidebar.markdown("- furnished -> 2")
st.sidebar.markdown("- semi-furnished -> 1")
st.sidebar.markdown("- unfurnished -> 0")

st.markdown(
    """
    <div class='hero'>
      <h2 style='margin:0;'>Estimateur de valeur immobiliere</h2>
      <p style='margin:0.4rem 0 0 0;'>
        Ajustez le profil d'un logement, lancez l'inference et visualisez immediatement
        une estimation coherente avec le modele enregistre dans Snowflake.
      </p>
    </div>
    """,
    unsafe_allow_html=True,
)

left_col, right_col = st.columns([1.15, 0.85], gap="large")

with left_col:
    st.markdown("<div class='panel'><div class='panel-title'>Caracteristiques du bien</div><div class='hint'>Saisissez les attributs structurels et les elements de confort du logement.</div></div>", unsafe_allow_html=True)
    area = st.slider("Surface (m2)", min_value=50, max_value=1000, value=150, step=5)
    bedrooms = st.slider("Chambres", min_value=1, max_value=10, value=3)
    bathrooms = st.slider("Salles de bain", min_value=1, max_value=5, value=2)
    stories = st.slider("Etages", min_value=1, max_value=5, value=2)
    parking = st.slider("Places de parking", min_value=0, max_value=5, value=1)

    st.markdown("<div class='panel'><div class='panel-title'>Environnement et equipements</div><div class='hint'>Ces champs alimentent directement le vecteur de prediction envoye au modele.</div></div>", unsafe_allow_html=True)
    c1, c2, c3 = st.columns(3)
    with c1:
        mainroad = st.selectbox("Route principale", ["yes", "no"])
        guestroom = st.selectbox("Chambre d'amis", ["yes", "no"])
        basement = st.selectbox("Sous-sol", ["yes", "no"])
    with c2:
        hotwater = st.selectbox("Chauffage a eau chaude", ["yes", "no"])
        aircon = st.selectbox("Climatisation", ["yes", "no"])
        prefarea = st.selectbox("Zone privilegiee", ["yes", "no"])
    with c3:
        furnished = st.selectbox("Ameublement", ["furnished", "semi-furnished", "unfurnished"])

with right_col:
    st.markdown("<div class='panel'><div class='panel-title'>Lecture du scenario</div><div class='hint'>Cette vue synthetise les entrees courantes avant l'estimation.</div></div>", unsafe_allow_html=True)

    comfort_score = sum([
        1 if mainroad == "yes" else 0,
        1 if guestroom == "yes" else 0,
        1 if basement == "yes" else 0,
        1 if hotwater == "yes" else 0,
        1 if aircon == "yes" else 0,
        1 if prefarea == "yes" else 0,
    ])

    scenario_tags = [
        f"Surface {area} m2",
        f"{bedrooms} chambres",
        f"{bathrooms} salles d'eau",
        f"{stories} niveaux",
        f"Parking {parking}",
        furnished,
    ]

    st.markdown("".join([f"<span class='badge'>{tag}</span>" for tag in scenario_tags]), unsafe_allow_html=True)
    st.progress(comfort_score / 6, text=f"Indice de confort : {comfort_score}/6")

    area_band = "Compact" if area < 120 else "Familial" if area < 260 else "Prestige"
    st.caption(f"Positionnement estime du bien : {area_band}")

def encode(val):
    return 1 if val == "yes" else 0

furnish_map = {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}

estimate_col, clear_col = st.columns([0.78, 0.22])
with estimate_col:
    run_prediction = st.button("Lancer l'estimation", use_container_width=True)
with clear_col:
    reset_view = st.button("Effacer", use_container_width=True)

if reset_view:
    st.session_state.pop("prediction_payload", None)
    st.rerun()

if run_prediction:
    input_data = pd.DataFrame([{
        "AREA": area,
        "BEDROOMS": bedrooms,
        "BATHROOMS": bathrooms,
        "STORIES": stories,
        "MAINROAD": encode(mainroad),
        "GUESTROOM": encode(guestroom),
        "BASEMENT": encode(basement),
        "HOTWATERHEATING": encode(hotwater),
        "AIRCONDITIONING": encode(aircon),
        "PARKING": parking,
        "PREFAREA": encode(prefarea),
        "FURNISHINGSTATUS": furnish_map[furnished],
    }]).astype("int16")

    with st.spinner("Calcul de la prediction en cours..."):
        registry = Registry(session=session)
        model_ref = registry.get_model("house_price_xgboost").version("v1")
        result = model_ref.run(input_data, function_name="predict")
        price = float(result["output_feature_0"].values[0])

    st.session_state["prediction_payload"] = {
        "price": price,
        "input_data": input_data,
        "ppsm": price / max(area, 1),
    }

if "prediction_payload" in st.session_state:
    payload = st.session_state["prediction_payload"]
    price = payload["price"]
    ppsm = payload["ppsm"]
    input_data = payload["input_data"]

    st.markdown("### Estimation calculee")
    st.markdown(
        f"""
        <div class='result-box'>
          <h3 style='margin:0;'>Valeur estimee : {price:,.0f} EUR</h3>
          <p style='margin:0.35rem 0 0 0;'>Valeur estimee au m2 : <strong>{ppsm:,.0f} EUR/m2</strong></p>
        </div>
        """,
        unsafe_allow_html=True,
    )

    m1, m2, m3 = st.columns(3)
    m1.metric("Prix estime", f"{price:,.0f} EUR")
    m2.metric("Prix au m2", f"{ppsm:,.0f} EUR/m2")
    m3.metric("Indice confort", f"{comfort_score}/6")

    confidence_proxy = min(max(comfort_score / 6, 0.15), 1.0)
    st.progress(confidence_proxy, text="Lecture de qualite du scenario")

    st.markdown("### Variables envoyees au modele")
    st.dataframe(input_data, use_container_width=True)
    st.success("Le scoring a ete execute avec succes.")
